<a href="https://colab.research.google.com/github/kavix/quiskit-question-01/blob/main/Quiskit01DeustchAlgorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# install relavant packages

In [76]:
!pip install qiskit matplotlib pyLatexenc

In [77]:
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler

print("Imported Successfully")

Imported Successfully


In [78]:
def exercise_1_superposition() -> QuantumCircuit:
  qc = QuantumCircuit(1,1)
  qc.h(0)
  qc.measure(0,0)
  return qc

qc1 = exercise_1_superposition()
print(qc1.draw())

     ┌───┐┌─┐
  q: ┤ H ├┤M├
     └───┘└╥┘
c: 1/══════╩═
           0 


In [79]:
sampler = StatevectorSampler()

counts1 = sampler.run([qc1],shots=1000).result()[0].data.c.get_counts()

print("Superposition Counts ",counts1)
assert "0" in counts1 and "1" in counts1 and 350<=counts1["0"] <=650, "check "
print("[PASS] Question 01 Complete")

Superposition Counts  {'0': 506, '1': 494}
[PASS] Question 01 Complete


## Question 02

In [80]:
def exercise_2_phase_kickback() -> QuantumCircuit:
  qc = QuantumCircuit(2,1)
  qc.x(1)
  qc.h(0)
  qc.h(1)
  qc.cx(0,1)
  qc.h(0)
  qc.measure(0,0)
  return qc

qc2 = exercise_2_phase_kickback()
print(qc2.draw())

     ┌───┐          ┌───┐┌─┐
q_0: ┤ H ├───────■──┤ H ├┤M├
     ├───┤┌───┐┌─┴─┐└───┘└╥┘
q_1: ┤ X ├┤ H ├┤ X ├──────╫─
     └───┘└───┘└───┘      ║ 
c: 1/═════════════════════╩═
                          0 


In [81]:
counts2 = StatevectorSampler().run([qc2],shots=128).result()[0].data.c.get_counts()
print("Phase Kickback Counts ",counts2)
assert counts2 == {"1":128}, f"Expected {{'1':128}}, got {counts2}"
print("[PASS] Question 02 Complete")

Phase Kickback Counts  {'1': 128}
[PASS] Question 02 Complete


SyntaxError: invalid syntax (3367047743.py, line 3)

# Question 3

In [85]:
def exercise_3_build_oracle(case:str) -> QuantumCircuit:
  oracle = QuantumCircuit(2,name=case)
  if case == "f0":
    pass
  elif case == "f1":
    oracle.x(1)
  elif case == "f2":
    oracle.cx(0,1)
  elif case == "f3":
    oracle.x(0)
    oracle.cx(0,1)
  else:
    raise ValueError(f"Unknown case {case}")
    oracle.to_gate()
  return oracle


for name in ["f0","f1","f2","f3"]:
  print("------- Oracle "+name+"-------")
  print(exercise_3_build_oracle(name).draw())

------- Oracle f0-------
     
q_0: 
     
q_1: 
     
------- Oracle f1-------
          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘
------- Oracle f2-------
          
q_0: ──■──
     ┌─┴─┐
q_1: ┤ X ├
     └───┘
------- Oracle f3-------
     ┌───┐     
q_0: ┤ X ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘


In [83]:
def exercise_4_deustch_algorithm(oracle:QuantumCircuit) -> QuantumCircuit:
  qc = QuantumCircuit(2,1)
  qc.x(1)
  qc.h(0)
  qc.h(1)
  qc.append(oracle,[0,1])
  qc.h(0)
  qc.measure(0,0)
  return qc

sampler = StatevectorSampler()

expected = {
    "f0": ("0","CONSTANT"),
    "f1": ("0","CONSTANT"),
    "f2": ("1","BALANCED"),
    "f3": ("1","BALANCED"),
}

for name, (expected_bit,label) in expected.items():
  oracle = exercise_3_build_oracle(name)
  qc = exercise_4_deustch_algorithm(oracle)
  counts = sampler.run([qc],shots=128).result()[0].data.c.get_counts()
  print(f"{name} -> counts: {counts} ({label})")
  assert counts == {expected_bit:128}, f"Expected {{'{expected_bit}':128}}, got {counts}"

print("[PASS] Question 04 Complete. All solve in One QUERY")

f0 -> counts: {'0': 128} (CONSTANT)
f1 -> counts: {'0': 128} (CONSTANT)
f2 -> counts: {'1': 128} (BALANCED)
f3 -> counts: {'1': 128} (BALANCED)
[PASS] Question 04 Complete. All solve in One QUERY


# Question 05 Oracle Classifier

In [84]:
def bonus_classify_mistry_oracle(oracle:QuantumCircuit) -> str:
  qc101 =  exercise_4_deustch_algorithm(oracle)
  sampler = StatevectorSampler()
  counts = sampler.run([qc101],shots=128).result()[0].data.c.get_counts()

  if counts.get("0",0) == 128:
    return "CONSTANT"
  elif counts.get("1",0) == 128:
    return "BALANCED"
  else:
    raise ValueError(f"Unknown counts {counts}")
  raise NotImplementedError("Complete bonus classifier mystry oracle Question")


for name, (_,expected_label) in expected.items():
  result_label = bonus_classify_mistry_oracle(exercise_3_build_oracle(name))
  assert result_label == expected_label, f"Expected {expected_label}, got {result_label}"

print("[PASS] Bonus Question Complete")

[PASS] Bonus Question Complete
